# 🎯 [TRN02_M07] 이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 검토 인터페이스 설계 (학생 실습용)
### Google Colab 실습 환경 (교육·훈련 / Level 2 - 학생 실습용)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/TRN02/trn02_m07_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 교육·훈련 (Level 2) |
| **미션 ID** | TRN02_M07 |
| **미션 주제** | 이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 검토 인터페이스 설계 |
| **소요 시간** | 약 3시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
미국-이란 전쟁 사례에서 보듯 드론과 미사일이 섞여 날아오는 전장에서는 장병 한 사람 한 사람이 비행체 형상을 식별하고, 아군·민간기와 구분하며, 정해진 보고·통제 절차를 지키는 능력이 방공의 기초가 된다. 그러나 부대 단위 방공 식별 교육은 채점과 오답 분석에 교관의 손이 많이 들어 개인별 약점을 제때 짚어 주기 어렵다. 본 프로젝트에서는 교육용 가상 문항 은행과 가상 응시자 응답을 코드로 만들어, 문항 데이터베이스 정의, 응답 정제·파싱, 자동 채점, 오답 유형(개념 혼동·피아 오인·절차 누락) 분류, 취약 영역 추천, 교수자 대시보드, 교관 검토(HITL), 대규모 동시 응시 부하 시험, AAR 문항 분석과 재보정, 최종 통합까지 채점·피드백 체계를 단계별로 구현한다. 최종 목표는 채점은 빠르고 정확하게 자동화하되, 이의 제기와 서술형 판단은 교관이 확정하는 신뢰할 수 있는 방공 교육 평가 도구를 만드는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 교육·훈련 평가체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/trn02_input_schema.json` | 문항 데이터베이스 스키마 및 가상 응답 데이터셋 규격 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_trn02.py` | 응답 전처리, 자동 채점 Baseline, 오답 유형 분류, 취약 영역 추천, 교수자 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/trn02_data_spec.md`, `docs/trn02_hitl_guide.md`, `docs/trn02_aar_report.md`, `docs/trn02_final_report.md` | 데이터 규격서, HITL 교관 검토 가이드, AAR 분석 보고서, 최종 운영 매뉴얼 등 기술/행정 문서 |
| **`configs/`**| `configs/trn02_tuned_params.yaml` | AAR 피드백 기반 재조정 변별도 파라미터 및 가중치 설정 파일 |
| **`tests/`** | `tests/trn02_stress_test.py` | 전 부대원 동시 응시 시나리오 대규모 트래픽 부하 테스트 및 최적화 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/...`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


---

## 🎯 미션 07. 이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 검토 인터페이스 설계
**소요시간**: 약 3.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 인터페이스를 설계한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **이의 제기 문항과 서술형 답변을 교관이 검토·확정하는 HITL 흐름을 설계하고, 자동 채점이 점수를 단정하지 못하게 한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **서술형은 핵심 요소 키워드로 점수를 '제안'만 하고, 이의 제기와 함께 합격선 근처·저신뢰 답안부터 검토하는 우선순위 대기열에 올린다 과정을 진행하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/hitl/trn02_approval_workflow.py 및 docs/trn02_hitl_guide.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 지휘 책임 분계선 명확화 및 성급한 자동 단정 방지 체계 부합성

#### 미션 PASS 조건

- 정량 기준: 인간 검토 큐(Queue) 대기시간 및 승인/반려 처리 로깅 성공률 100%


- 진행 기준: 승인/반려 인터페이스 동작 및 감사 로그(Audit Log) 생성 검증


- 검증 방법: 모의 조작자 승인/반려 시뮬레이션 및 데이터 무결성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""이의 제기된 단답형 문항과 서술형 답변을 교관 검토 대기열(HITL)에 올려, 자동 채점은 '제안'으로만 두고 교관의 인정·기각·수정 결정과 사유를 해시 체인 감사 로그에 남기는 검토 흐름을 설계한다. 합격선 근처 응시자를 먼저 검토하는 우선순위 큐의 대기시간, 로깅 성공률, 무결성 검증을 확인한다."""
import difflib
import hashlib
import heapq
import json
import re
import unicodedata
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# [교육용가상자료] 방공·드론 식별 퀴즈 문항 은행과 가상 응시자 응답 생성기
# 문항은 일반 원칙 수준의 가상 문항이며 실제 교범·교전규칙 원문이 아니다. 응시자는 가명(S001~)이다.
# ---------------------------------------------------------------------------
AREAS = ["형상식별", "피아식별", "경보보고", "교전통제", "대드론일반"]
ETYPES = ["개념 혼동", "피아 오인", "절차 누락"]
POINTS = {1: 2, 2: 3, 3: 5}
C, F, P = ETYPES


def _mc(qid, area, diff, stem, opts, ans, dist):
    # [TODO : 학생 실습 7] 이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 검토 인터페이스 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sa(qid, area, diff, stem, accept, wrong):
    # [TODO : 학생 실습 7] 이의 제기 문항 및 서술형 답변에 대한 교관 수동 채점(HITL) 검토 인터페이스 설계 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _order(qid, area, diff, stem):
    return {"qid": qid, "type": "ORDER", "area": area, "diff": diff, "stem": stem, "answer": "ABCD"}


BANK = [
    _mc("Q01", "형상식별", 1, "꼬리날개 없이 삼각형 날개를 가진 소형 고정익 실루엣은?", ["쿼드콥터형", "삼각익 고정익형", "회전익 헬기형", "조류"], 2, [C, C, F]),
    _mc("Q02", "형상식별", 1, "회전날개 4개가 십자로 배치된 실루엣은?", ["고정익 드론형", "쿼드콥터형", "순항미사일형", "탄도미사일형"], 2, [C, C, C]),
    _mc("Q03", "형상식별", 2, "가늘고 긴 동체 중간에 작은 날개가 붙은 실루엣은?", ["순항미사일형", "탄도미사일형", "민간 경비행기형", "조류"], 1, [C, F, F]),
    _mc("Q04", "형상식별", 3, "원거리라 실루엣 식별이 어려울 때 먼저 할 행동은?", ["추정 유형으로 확정 보고", "관측 정보를 그대로 보고하고 추가 관측", "관측 중단", "민간기로 간주"], 2, [P, P, F]),
    _mc("Q05", "피아식별", 2, "식별 신호 응답이 없는 항공기에 대한 올바른 판단은?", ["즉시 적기로 확정", "미확인 항공기로 분류하고 추가 확인", "아군기로 간주", "무시"], 2, [F, F, P]),
    _mc("Q06", "피아식별", 2, "운항 정보와 일치하는 저속 비행체의 분류는?", ["적 드론", "민간기 가능성 높음, 추가 확인", "순항미사일", "아군 정찰기로 확정"], 2, [F, C, F]),
    _mc("Q07", "피아식별", 3, "아군 드론 운용 공지가 있는 공역에서 미확인 드론을 보면 우선 확인할 것은?", ["기상 정보", "아군 운용 계획과 식별 신호", "대응 준비 상태", "주변 지형"], 2, [C, P, C]),
    _mc("Q08", "피아식별", 3, "조류 무리와 소형 드론 군집 구분에 가장 도움이 되는 정보는?", ["크기만 비교", "궤적의 규칙성과 열 신호", "색상", "관측 시각"], 2, [C, C, C]),
    _mc("Q09", "경보보고", 1, "공중 위협 최초 보고에 포함하지 않아도 되는 것은?", ["관측 시각", "방향과 거리", "추정 유형", "관측자 개인 연락처"], 4, [P, P, P]),
    _mc("Q10", "경보보고", 2, "경보 전파 후 상황이 바뀌면?", ["최초 보고 유지", "변경 사항을 즉시 추가 보고", "상황 종료 후 일괄 보고", "개인 판단으로 경보 해제"], 2, [P, P, P]),
    _mc("Q11", "경보보고", 2, "보고 내용이 불확실할 때 올바른 표현은?", ["확정 표현으로 보고", "'추정'임을 밝히고 근거와 함께 보고", "보고 보류", "다른 관측자 보고를 복사"], 2, [C, P, P]),
    _mc("Q12", "경보보고", 1, "드론 탐지 후 보고 경로로 알맞은 것은?", ["인접 부대에만 전파", "지휘계통을 따라 보고", "개인 메신저로 공유", "보고 없이 대응"], 2, [P, P, P]),
    _mc("Q13", "교전통제", 1, "AI 탐지 결과만으로 대응을 결정할 수 있는가?", ["가능하다", "인간 지휘관의 승인이 필요하다", "신뢰도 90% 이상이면 가능", "야간에만 가능"], 2, [C, C, C]),
    _mc("Q14", "교전통제", 2, "민간 피해 가능성이 확인되면?", ["그대로 진행", "즉시 보고하고 지휘관 판단을 받는다", "조치 후 보고", "무시"], 2, [P, P, F]),
    _mc("Q15", "교전통제", 3, "식별이 불확실한 표적에 대한 원칙은?", ["위협으로 간주해 즉시 대응", "식별 확인 전까지 승인 절차를 따른다", "무조건 무시", "다수결로 결정"], 2, [F, F, C]),
    _mc("Q16", "교전통제", 2, "판단 근거를 기록해야 하는 이유는?", ["형식상 필요", "사후 검토와 책임 추적", "보고 생략", "기록 불필요"], 2, [C, P, P]),
    _mc("Q17", "대드론일반", 1, "대드론 체계의 일반적 처리 순서는?", ["대응-탐지-식별", "탐지-식별-보고·판단-대응", "식별-대응-탐지", "보고-탐지-대응"], 2, [P, P, P]),
    _mc("Q18", "대드론일반", 3, "레이더만으로 조류와 드론 구분이 어려운 이유는?", ["반사 신호 크기가 비슷할 수 있어서", "레이더는 야간에 못 써서", "드론에는 금속이 없어서", "조류는 탐지되지 않아서"], 1, [C, C, C]),
    _mc("Q19", "대드론일반", 2, "전자광학(EO) 센서의 약점이 커지는 조건은?", ["맑은 주간", "안개·야간", "저속 표적", "근거리"], 2, [C, C, C]),
    _mc("Q20", "대드론일반", 2, "경보가 반복해서 오경보로 판명되면?", ["경보 장치를 끈다", "사례를 기록·보고하고 기준 재검토 요청", "무시한다", "모두 실제 위협으로 간주"], 2, [P, P, F]),
    _sa("Q21", "피아식별", 1, "아군·적·민간을 구분하는 절차를 무엇이라 하는가?", ["피아식별", "IFF"], {"표적획득": C, "적기판정": F, "경계": C}),
    _sa("Q22", "형상식별", 2, "꼬리날개 없는 삼각형 날개 형태를 무엇이라 하는가?", ["삼각익", "델타익", "delta"], {"직선익": C, "후퇴익": C, "쿼드콥터": C}),
    _sa("Q23", "경보보고", 2, "불확실한 관측을 보고할 때 붙이는 표현은?", ["추정"], {"확정": C, "확인": C, "생략": P}),
    _sa("Q24", "교전통제", 1, "AI 탐지 결과에 대해 최종 결정을 내리는 주체는?", ["지휘관"], {"AI": C, "운용병": P, "레이더": C}),
    _sa("Q25", "대드론일반", 1, "조류를 드론으로 잘못 경보하는 것을 무엇이라 하는가?", ["오경보", "오탐", "false alarm"], {"미탐": C, "재경보": C, "적기": F}),
    _sa("Q26", "피아식별", 2, "운항 정보가 확인된 여객기는 어떻게 분류하는가?", ["민간기", "민항기"], {"적기": F, "미확인기": P, "아군기": F}),
    _sa("Q27", "형상식별", 1, "회전날개 4개를 가진 드론 형태는?", ["쿼드콥터", "quadcopter"], {"헥사콥터": C, "고정익": C, "헬기": F}),
    _sa("Q28", "경보보고", 1, "보고는 무엇을 따라 올라가는가?", ["지휘계통"], {"인접부대": P, "개인메신저": P, "통신망": C}),
    _order("Q29", "경보보고", 2, "최초 보고 순서: A 관측 시각, B 방향·거리, C 추정 유형, D 관측자 조치"),
    _order("Q30", "대드론일반", 2, "처리 순서: A 탐지, B 식별, C 보고·판단, D 대응"),
    _order("Q31", "교전통제", 3, "통제 순서: A 식별 확인, B 상급 승인 요청, C 승인 수령, D 조치 기록"),
    _order("Q32", "피아식별", 3, "확인 순서: A 운항정보 대조, B 식별 신호 확인, C 미확인 분류, D 보고"),
]
QIDX = {q["qid"]: q for q in BANK}
UNITS = ["가상1중대", "가상2중대", "가상3중대"]
_CIRC = "①②③④"
PARA = {"Q21": "아군적군 구분", "Q22": "세모 날개", "Q23": "추정됨으로 표기", "Q24": "중대장(지휘관)", "Q25": "잘못된 경보",
        "Q26": "민간 여객기", "Q27": "4발 드론", "Q28": "지휘 라인"}          # 뜻은 맞지만 인정 답안 목록에 없는 표현


def make_soldiers(n_per_unit, seed):
    rng = np.random.default_rng(seed)
    out = []
    for u, unit in enumerate(UNITS):
        base = [0.3, -0.1, 0.0][u]
        for i in range(n_per_unit):
            off = rng.normal(0, 0.5, len(AREAS)) + (np.array([0, 0, 0, -0.6, 0]) if u == 1 else 0)
            out.append({"sid": f"S{len(out) + 1:03d}", "unit": unit, "theta": dict(zip(AREAS, base + rng.normal(0, 0.6) + off)),
                        "tend": dict(zip(ETYPES, rng.dirichlet([1.2, 1.2, 1.2])))})
    return out


def p_correct(theta, diff):
    return 1 / (1 + np.exp(-1.7 * (theta - (diff - 2) * 0.9 + 0.9)))


def _typo(s, rng):
    if len(s) < 2:
        return s
    i = int(rng.integers(len(s) - 1))
    return s[:i] + s[i + 1] + s[i] + s[i + 2:] if len(s) <= 3 or rng.random() < 0.5 else s[:i] + s[i + 1:]


def _fmt_mc(k, rng):
    r = rng.random()
    if r < 0.55:
        return str(k)
    return rng.choice([_CIRC[k - 1], f"{k}번", "ABCD"[k - 1], "abcd"[k - 1] + ")", f" {k} ", chr(0xFF10 + k), f"{k}."])


def _fmt_sa(s, rng):
    r = rng.random()
    if r < 0.45:
        return s
    if r < 0.6:
        return " ".join(s) if len(s) <= 4 else s[:2] + " " + s[2:]
    if r < 0.72:
        return s + rng.choice(["입니다", "이다", ".", "임"])
    if r < 0.8:
        return s.upper() if s.isascii() else "".join(chr(ord(ch) + 0xFEE0) if "!" <= ch <= "~" else ch for ch in s) + " "
    return _typo(s, rng)


def answer(q, sol, rng):
    """한 문항 응답: (원본 응답 문자열, 의도한 정답 여부, 의도한 답, 오답 유형)."""
    ok = rng.random() < p_correct(sol["theta"][q["area"]], q["diff"])
    if q["type"] == "MC":
        if ok:
            k, et = q["answer"], None
        else:
            ks = list(q["dist"]); w = np.array([sol["tend"][q["dist"][k]] for k in ks])
            k = int(rng.choice(ks, p=w / w.sum())); et = q["dist"][k]
        return _fmt_mc(k, rng), ok, str(k), et
    if q["type"] == "SA":
        if ok and rng.random() < 0.06:
            return PARA[q["qid"]], ok, PARA[q["qid"]], None
        if ok:
            s, et = str(rng.choice(q["accept"])), None
        else:
            ws = list(q["wrong"]); w = np.array([sol["tend"][q["wrong"][x]] for x in ws])
            s = str(rng.choice(ws, p=w / w.sum())); et = q["wrong"][s]
        return _fmt_sa(s, rng), ok, s, et
    if ok:
        s = "ABCD"
    elif rng.random() < 0.5:
        s = "ABCD".replace("ABCD"[rng.integers(1, 4)], "")       # 단계 하나 빠뜨림
    else:
        i = int(rng.integers(0, 3)); s = list("ABCD"); s[i], s[i + 1] = s[i + 1], s[i]; s = "".join(s)   # 순서 뒤바뀜
    raw = s if rng.random() < 0.6 else rng.choice(["-".join(s), " ".join(s), s.lower(), ",".join(s)])
    return raw, ok, s, (None if ok else P)


def make_responses(n_per_unit=40, seed=7, blank=0.02):
    """응시자 x 문항 응답표(DataFrame)를 만든다. truth_* 열은 교관 기준 정답(평가용)이다."""
    rng = np.random.default_rng(seed)
    rows = []
    for sol in make_soldiers(n_per_unit, seed):
        for q in BANK:
            raw, ok, intended, et = answer(q, sol, rng)
            if rng.random() < blank:
                raw, ok, intended, et = "", False, "", None
            rows.append({"rid": f"R{len(rows):05d}", "sid": sol["sid"], "unit": sol["unit"], "qid": q["qid"], "raw": raw,
                         "truth_correct": ok, "truth_answer": intended, "truth_etype": et})
    return pd.DataFrame(rows)
# ---------------------------------------------------------------------------

# ---- 응답 정제·파싱·채점 공통 함수 ----
TAILS = ("입니다", "이다", "임")


def norm_text(s):
    """전각->반각(NFKC), 소문자, 공백·문장부호 제거, 끝맺음 어미 제거."""
    s = unicodedata.normalize("NFKC", str(s)).strip().lower()
    s = re.sub(r"[\s\.\,\-\(\)·]", "", s)
    for t in TAILS:
        if s.endswith(t) and len(s) > len(t) + 1:
            s = s[:-len(t)]
    return s


def parse(q, raw):
    """원본 응답 -> (정제값, 상태). 상태: 정상 / 미응답 / 형식오류"""
    if raw is None or str(raw).strip() == "":
        return "", "미응답"
    s = unicodedata.normalize("NFKC", str(raw)).strip()
    if q["type"] == "MC":
        t = re.sub(r"[\s번\.\)]", "", s).lower()
        t = {"a": "1", "b": "2", "c": "3", "d": "4"}.get(t, t)
        return (t, "정상") if t in ("1", "2", "3", "4") else (t, "형식오류")
    if q["type"] == "ORDER":
        t = re.sub(r"[^A-Za-z]", "", s).upper()
        return (t, "정상") if t and set(t) <= set("ABCD") and len(set(t)) == len(t) else (t, "형식오류")
    return norm_text(s), "정상"


def grade_exact(q, val):
    if q["type"] == "MC":
        return val == str(q["answer"])
    if q["type"] == "ORDER":
        return val == q["answer"]
    return val in {norm_text(a) for a in q["accept"]}


def grade_fuzzy(q, val, cut=0.75):
    """단답형은 오탈자(글자 순서 바뀜·한 글자 누락)를 허용(유사도 cut 이상)하되, 알려진 오답과 더 가까우면 오답으로 본다."""
    if grade_exact(q, val) or q["type"] != "SA" or len(val) < 2:
        return grade_exact(q, val)
    best = max(1.0 if sorted(val) == sorted(norm_text(a)) else difflib.SequenceMatcher(None, val, norm_text(a)).ratio() for a in q["accept"])
    worst = max(difflib.SequenceMatcher(None, val, norm_text(w)).ratio() for w in q["wrong"])
    return best >= cut and best > worst


def score_of(q, correct):
    return POINTS[q["diff"]] if correct else 0

# ---- 서술형 문항(10점): 핵심 요소 5개 x 2점 ----
ESSAY = "미확인 드론을 관측했을 때 보고와 대응 절차를 서술하시오."
ELEM = {"관측정보": (["관측 시각과 방향, 거리를 적는다", "언제 어느 방향 몇 km인지 기록한다"], ["시각", "방향", "거리"]),
        "추정표현": (["확실하지 않으면 추정이라고 밝힌다", "모르는 부분은 추정임을 명시한다"], ["추정"]),
        "지휘계통": (["지휘계통을 따라 즉시 보고한다", "소대장에게 보고하고 상급으로 올린다"], ["지휘계통"]),
        "승인대기": (["지휘관 승인 전에는 대응하지 않는다", "승인이 나기 전까지 조치하지 않고 대기한다"], ["승인"]),
        "근거기록": (["판단 근거를 기록한다", "조치 내용을 일지에 남긴다"], ["기록", "근거"])}
WRONG_SENT = ["승인 없이 바로 대응한다", "기록은 상황 종료 후 생략한다", "확정된 적 드론으로 보고한다"]   # 키워드는 있지만 틀린 진술


def make_essays(sids, seed):
    rng = np.random.default_rng(seed)
    out = []
    for s in sids:
        have = [e for e in ELEM if rng.random() < 0.65]
        sents = [str(rng.choice(ELEM[e][0])) for e in have]
        if rng.random() < 0.2:
            sents.append(str(rng.choice(WRONG_SENT)))
        rng.shuffle(sents)
        out.append({"sid": s, "text": " ".join(sents), "truth": 2 * len(have)})
    return out


def auto_suggest(text):
    """키워드 포함 여부로 점수를 '제안'한다(확정 아님). 부정 문맥을 구분하지 못하는 한계가 있다."""
    hit = [e for e, (_, kws) in ELEM.items() if any(k in text for k in kws)]
    conf = "낮음" if any(w in text for w in ("없이", "생략", "확정")) or len(text) < 25 else "보통"
    return 2 * len(hit), hit, conf


class ReviewQueue:
    """우선순위 큐 + 해시 체인 감사 로그. 교관 결정 없이는 점수를 확정할 수 없다."""
    def __init__(self):
        self.heap, self.items, self.log, self.final = [], {}, [], {}

    def _audit(self, event, **kw):
        prev = self.log[-1]["hash"] if self.log else "GENESIS"
        body = {"seq": len(self.log), "event": event, **kw}
        self.log.append({**body, "prev": prev, "hash": hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest()})

    def submit(self, iid, prio, t, payload):
        self.items[iid] = {**payload, "t_in": t, "status": "PENDING"}
        heapq.heappush(self.heap, (prio, t, iid))
        self._audit("SUBMIT", iid=iid, prio=prio, suggest=payload.get("suggest"))

    def pop(self):
        return heapq.heappop(self.heap)[2] if self.heap else None

    def decide(self, iid, reviewer, decision, score, reason, t):
        if not reason:
            raise ValueError("사유 없는 결정은 기록할 수 없다")
        it = self.items[iid]
        it.update(status=decision, score=score, reviewer=reviewer, t_out=t)
        self.final[iid] = score
        self._audit("DECIDE", iid=iid, reviewer=reviewer, decision=decision, score=score, reason=reason)

    def finalize_without_review(self, iid):
        if self.items[iid]["status"] == "PENDING":
            raise PermissionError("교관 결정 전 자동 확정 금지")

    def verify(self):
        prev = "GENESIS"
        for e in self.log:
            body = {k: v for k, v in e.items() if k not in ("prev", "hash")}
            if e["prev"] != prev or hashlib.sha256((prev + json.dumps(body, sort_keys=True, ensure_ascii=False)).encode()).hexdigest() != e["hash"]:
                return False, e["seq"]
            prev = e["hash"]
        return True, None


rng = np.random.default_rng(3)
df = make_responses(40, seed=7)
df["val"] = [parse(QIDX[q], r)[0] for q, r in zip(df["qid"], df["raw"])]
df["correct"] = [grade_fuzzy(QIDX[q], v) for q, v in zip(df["qid"], df["val"])]
df["score"] = [score_of(QIDX[q], c) for q, c in zip(df["qid"], df["correct"])]
total = df.groupby("sid")["score"].sum()
PASS = 0.7 * (sum(POINTS[q["diff"]] for q in BANK) + 10)

# 1) 이의 제기: 자동 채점 오답 처리된 단답형 중 응시자가 이의를 낸 건(실제 맞은 답은 대부분 이의 제기, 틀린 답도 일부 제기)
sa_wrong = df[(df["qid"].map(lambda q: QIDX[q]["type"]) == "SA") & ~df["correct"] & (df["raw"] != "")]
appeal = sa_wrong[sa_wrong["truth_correct"] | (rng.random(len(sa_wrong)) < 0.08)]
essays = make_essays(sorted(df["sid"].unique()), seed=5)
print(f"[교육용가상자료] 이의 제기 {len(appeal)}건, 서술형 답안 {len(essays)}건이 교관 검토 대기열에 들어간다.")

q = ReviewQueue()
t = 0.0
for e in essays:
    sug, hit, conf = auto_suggest(e["text"])
    est = total[e["sid"]] + sug
    prio = (0 if abs(est - PASS) <= 6 else 1) * 10 + (0 if conf == "낮음" else 1)   # 합격선 근처·저신뢰 먼저
    q.submit(f"E-{e['sid']}", prio, t, {"kind": "서술형", "sid": e["sid"], "suggest": sug, "conf": conf, "truth": e["truth"], "text": e["text"]})
    t += 0.05
for _, r in appeal.iterrows():
    est = total[r["sid"]]
    prio = (0 if abs(est - PASS) <= 6 else 1) * 10 + 2
    q.submit(f"A-{r['rid']}", prio, t, {"kind": "이의 제기", "sid": r["sid"], "qid": r["qid"], "raw": r["raw"], "suggest": 0,
                                         "truth": score_of(QIDX[r["qid"]], r["truth_correct"])})
    t += 0.05

# 2) 자동 확정 시도는 차단되어야 한다
blocked = 0
for iid in list(q.items)[:5]:
    try:
        q.finalize_without_review(iid)
    except PermissionError:
        blocked += 1
try:
    q.decide(list(q.items)[0], "교관A", "인정", 2, "", t)
    no_reason = "통과(오류)"
except ValueError:
    no_reason = "차단"
print(f"결정 전 자동 확정 시도 5건 중 차단 {blocked}건, 사유 없는 결정 {no_reason}")

# 3) 교관 2명이 우선순위대로 검토(모의 교관은 기준 답안으로 판단: 건당 서술형 2분, 이의 제기 1분)
clock = {"교관A": 0.0, "교관B": 0.0}
while True:
    iid = q.pop()
    if iid is None:
        break
    it = q.items[iid]
    who = min(clock, key=clock.get)
    clock[who] = max(clock[who], it["t_in"]) + (2.0 if it["kind"] == "서술형" else 1.0)
    if it["kind"] == "서술형":
        dec = "승인" if it["truth"] == it["suggest"] else "수정"
        reason = "제안 점수와 요소 판단 일치" if dec == "승인" else f"요소 재판단(제안 {it['suggest']} -> {it['truth']})"
    else:
        dec = "인정" if it["truth"] > 0 else "기각"
        reason = f"'{it['raw']}' 의미상 정답 인정" if dec == "인정" else f"'{it['raw']}'는 오답 유지"
    q.decide(iid, who, dec, it["truth"], reason, clock[who])

R = pd.DataFrame(q.items.values())
R["wait"] = R["t_out"] - R["t_in"]
R["border"] = [abs(total[s] + (sg if k == "서술형" else 0) - PASS) <= 6 for s, sg, k in zip(R["sid"], R["suggest"], R["kind"])]
print("\n[검토 결과]")
print(R.groupby(["kind", "status"]).size().to_string())
ess = R[R["kind"] == "서술형"]
print(f"서술형 자동 제안 점수와 교관 판단 일치 {(ess['status'] == '승인').mean():.1%}, 수정 {(ess['status'] == '수정').sum()}건"
      f"(틀린 진술 속 키워드에 속은 제안 포함)")
print(f"대기시간(분): 합격선 근처 평균 {R.loc[R['border'], 'wait'].mean():.1f}, 그 외 평균 {R.loc[~R['border'], 'wait'].mean():.1f}, 최대 {R['wait'].max():.1f}")

n_dec = sum(e["event"] == "DECIDE" for e in q.log)
print(f"승인/반려 처리 {len(R)}건, 감사 로그 기록 {n_dec}건, 로깅 성공률 {n_dec / len(R):.0%} (목표 100%: {'달성' if n_dec == len(R) else '미달'})")
ok, _ = q.verify()
q.log[7]["score"] = 10                           # 감사 로그 사후 변조 시도
ok2, where = q.verify()
print(f"무결성 검증: 원본 {'정상' if ok else '이상'}, 변조 후 {'정상' if ok2 else f'이상 탐지(seq {where})'}")

changed = R[(R["kind"] == "이의 제기") & (R["status"] == "인정")].groupby("sid")["score"].sum()
newtot = total.add(changed, fill_value=0) + ess.set_index("sid")["score"]
flip = int(((total + ess.set_index("sid")["suggest"] >= PASS) != (newtot >= PASS)).sum())
print(f"교관 검토로 합격/불합격이 바뀐 응시자 {flip}명 -> 자동 채점만으로 확정하면 안 되는 이유")
print("\n[HITL 검토 지침 요약] 1) 자동 점수는 제안일 뿐 교관이 확정 2) 인정·기각·수정 모두 사유 필수"
      " 3) 합격선 근처·저신뢰 답안 우선 4) 감사 로그는 수정 불가, 이의 재심은 상급 교관(지휘 책임선)이 판단")


<details>
<summary><b>💡 [참고] 미션 07 예상 실행 결과 (Output)</b></summary>

```text
[교육용가상자료] 이의 제기 45건, 서술형 답안 120건이 교관 검토 대기열에 들어간다.
결정 전 자동 확정 시도 5건 중 차단 5건, 사유 없는 결정 차단

[검토 결과]
kind   status
서술형    수정        54
       승인        66
이의 제기  기각         7
       인정        38
서술형 자동 제안 점수와 교관 판단 일치 55.0%, 수정 54건(틀린 진술 속 키워드에 속은 제안 포함)
대기시간(분): 합격선 근처 평균 20.0, 그 외 평균 94.7, 최대 137.4
승인/반려 처리 165건, 감사 로그 기록 165건, 로깅 성공률 100% (목표 100%: 달성)
무결성 검증: 원본 정상, 변조 후 이상 탐지(seq 7)
교관 검토로 합격/불합격이 바뀐 응시자 3명 -> 자동 채점만으로 확정하면 안 되는 이유

[HITL 검토 지침 요약] 1) 자동 점수는 제안일 뿐 교관이 확정 2) 인정·기각·수정 모두 사유 필수 3) 합격선 근처·저신뢰 답안 우선 4) 감사 로그는 수정 불가, 이의 재심은 상급 교관(지휘 책임선)이 판단
```
</details>
